# Parte Específica: EDA estrutural do BankChurners

Dataset: Credit Card Customers (BankChurners), arquivo `data/parte_especifica/BankChurners.csv`
(já sem as duas colunas `Naive_Bayes_Classifier_...`).

Pilar estrutural: dimensões, tipos das variáveis e qualidade dos dados.

In [1]:
# [IA] Claude Code – importacao das bibliotecas e leitura do dataset
from pathlib import Path

import pandas as pd

CAMINHO_DADOS = Path("../../data/parte_especifica/BankChurners.csv")
df = pd.read_csv(CAMINHO_DADOS)
df.head()

,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000


## 1. Dimensões

In [2]:
# [IA] Claude Code – quantidade de linhas e colunas
linhas, colunas = df.shape
print(f"Linhas: {linhas}")
print(f"Colunas: {colunas}")

Linhas: 10127
Colunas: 21


## 2. Tipos das variáveis

In [3]:
# [IA] Claude Code – tabela de tipos: dtype do pandas, quantidade de valores distintos e exemplos
tipos = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "qtd_valores_distintos": df.nunique(),
    "exemplos": [df[c].drop_duplicates().head(4).tolist() for c in df.columns],
})
tipos["grupo"] = ["numerica" if pd.api.types.is_numeric_dtype(df[c]) else "categorica" for c in df.columns]
tipos

,dtype,qtd_valores_distintos,exemplos,grupo
CLIENTNUM,int64,10127,"[768805383, 818770008, 713982108, 769911858]",numerica
Attrition_Flag,str,2,"[Existing Customer, Attrited Customer]",categorica
Customer_Age,int64,45,"[45, 49, 51, 40]",numerica
Gender,str,2,"[M, F]",categorica
Dependent_count,int64,6,"[3, 5, 4, 2]",numerica
Education_Level,str,7,"[High School, Graduate, Uneducated, Unknown]",categorica
Marital_Status,str,4,"[Married, Single, Unknown, Divorced]",categorica
Income_Category,str,6,"[$60K - $80K, Less than $40K, $80K - $120K, $4...",categorica
Card_Category,str,4,"[Blue, Gold, Silver, Platinum]",categorica
Months_on_book,int64,44,"[39, 44, 36, 34]",numerica


In [4]:
# [IA] Claude Code – resumo da quantidade de variaveis por grupo
tipos["grupo"].value_counts()

grupo
numerica      15
categorica     6
Name: count, dtype: int64

### Classificação das variáveis

Adotamos a classificação das variáveis apresentada no dicionário do dataset, considerando seus significados e seus tipos estatísticos.


## 3. Qualidade dos dados

In [5]:
# [IA] Claude Code – valores ausentes (NaN) por coluna
ausentes = df.isna().sum()
print("Total de valores ausentes (NaN):", ausentes.sum())
ausentes[ausentes > 0]

Total de valores ausentes (NaN): 0


Series([], dtype: int64)

In [6]:
# [IA] Claude Code – linhas duplicadas e unicidade do identificador do cliente
print("Linhas duplicadas:", df.duplicated().sum())
print("CLIENTNUM repetidos:", df["CLIENTNUM"].duplicated().sum())

Linhas duplicadas: 0
CLIENTNUM repetidos: 0


In [7]:
# [IA] Claude Code – valores 'Unknown' nas variaveis categoricas (ausencia registrada como texto, nao como NaN)
colunas_categoricas = df.select_dtypes(exclude="number").columns

unknown = pd.DataFrame({
    "qtd_unknown": [(df[c] == "Unknown").sum() for c in colunas_categoricas],
}, index=colunas_categoricas)
unknown["pct_unknown"] = (unknown["qtd_unknown"] / len(df) * 100).round(2)
unknown.sort_values("qtd_unknown", ascending=False)

,qtd_unknown,pct_unknown
Education_Level,1519,15.00
Income_Category,1112,10.98
Marital_Status,749,7.40
Attrition_Flag,0,0.00
Gender,0,0.00
Card_Category,0,0.00


In [8]:
# [IA] Claude Code – linhas com pelo menos um 'Unknown' e quantos 'Unknown' cada linha tem
qtd_unknown_por_linha = (df[colunas_categoricas] == "Unknown").sum(axis=1)
print("Linhas com pelo menos um Unknown:", (qtd_unknown_por_linha > 0).sum(),
      f"({(qtd_unknown_por_linha > 0).mean() * 100:.2f}%)")
qtd_unknown_por_linha.value_counts().sort_index().rename_axis("qtd_unknown_na_linha").rename("qtd_linhas")

Linhas com pelo menos um Unknown: 3046 (30.08%)


qtd_unknown_na_linha
0    7081
1    2719
2     320
3       7
Name: qtd_linhas, dtype: int64

In [9]:
# [IA] Claude Code – categorias de cada variavel categorica (procura de grafias inconsistentes)
for coluna in colunas_categoricas:
    print(f"{coluna}: {df[coluna].value_counts().to_dict()}")

Attrition_Flag: {'Existing Customer': 8500, 'Attrited Customer': 1627}
Gender: {'F': 5358, 'M': 4769}
Education_Level: {'Graduate': 3128, 'High School': 2013, 'Unknown': 1519, 'Uneducated': 1487, 'College': 1013, 'Post-Graduate': 516, 'Doctorate': 451}
Marital_Status: {'Married': 4687, 'Single': 3943, 'Unknown': 749, 'Divorced': 748}
Income_Category: {'Less than $40K': 3561, '$40K - $60K': 1790, '$80K - $120K': 1535, '$60K - $80K': 1402, 'Unknown': 1112, '$120K +': 727}
Card_Category: {'Blue': 9436, 'Silver': 555, 'Gold': 116, 'Platinum': 20}


In [10]:
# [IA] Claude Code – faixas das variaveis numericas (procura de valores impossiveis, ex.: negativos)
colunas_numericas = df.select_dtypes(include="number").columns.drop("CLIENTNUM")

faixas = df[colunas_numericas].describe().T[["min", "max", "mean", "std"]].round(3)
faixas["qtd_negativos"] = (df[colunas_numericas] < 0).sum()
faixas

,min,max,mean,std,qtd_negativos
Customer_Age,26.0,73.000,46.326,8.017,0
Dependent_count,0.0,5.000,2.346,1.299,0
Months_on_book,13.0,56.000,35.928,7.986,0
Total_Relationship_Count,1.0,6.000,3.813,1.554,0
Months_Inactive_12_mon,0.0,6.000,2.341,1.011,0
Contacts_Count_12_mon,0.0,6.000,2.455,1.106,0
Credit_Limit,1438.3,34516.000,8631.954,9088.777,0
Total_Revolving_Bal,0.0,2517.000,1162.814,814.987,0
Avg_Open_To_Buy,3.0,34516.000,7469.140,9090.685,0
Total_Amt_Chng_Q4_Q1,0.0,3.397,0.760,0.219,0


## 4. Variável alvo

In [11]:
# [IA] Claude Code – contagem e proporcao das classes da variavel alvo
alvo = pd.DataFrame({
    "qtd": df["Attrition_Flag"].value_counts(),
    "pct": (df["Attrition_Flag"].value_counts(normalize=True) * 100).round(2),
})
alvo

,qtd,pct
Attrition_Flag,,
Existing Customer,8500,83.93
Attrited Customer,1627,16.07
